In [ ]:
from typing import TypedDict, Any, Annotated
import operator


class DataDescriptionState(TypedDict, total=False):
    # Input from main orchestrator
    dataset_path: str
    target_column: str | None

    # Optional config
    artifacts_dir: str

    # Stage results
    basic_overview: dict[str, Any]
    domain_understanding: dict[str, Any]
    schema_detection: dict[str, Any]
    data_quality: dict[str, Any]
    statistics: dict[str, Any]

    # Final compact result of Data Description Agent
    final_result: dict[str, Any]

    # Output for the next agent
    next_input: dict[str, Any]

    # Key detected task info
    task_type: str | None
    schema: dict[str, Any]
    data_summary: dict[str, Any]

    # Artifacts and logs
    artifacts: dict[str, str]
    logs: Annotated[list[dict[str, Any]], operator.add]

    # Status
    status: str
    error: str | None

In [ ]:
DATA_DESCRIPTION_ROLE_PROMPT = """
Ты — детерминированный агент анализа данных.

Твоя роль:
- Строго следовать инструкциям.
- Писать корректный и минимальный Python-код при использовании python_interpreter_tool.
- Возвращать ТОЛЬКО валидный JSON, когда запрошен JSON.
- Ничего не объяснять, когда запрошен JSON.
- Не добавлять markdown, комментарии, отладочный текст или любой лишний текст, когда запрошен JSON.
- Не придумывать колонки, значения, метрики или выводы.
- Опираться только на фактический датасет, вычисленные результаты и результаты предыдущих этапов.
- Если что-то непонятно, делать самое безопасное консервативное предположение и записывать его в notes, если схема ответа это позволяет.

Правила:
- Вывод должен быть только компактным валидным JSON.
- Никакого текста до или после JSON.
- Без markdown.
- Без объяснений.
""".strip()


def run_stage_with_llm(stage_name, task):
    print(f"\n========== START STAGE: {stage_name} ==========")

    try:
        result = data_description_stage_agent.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": DATA_DESCRIPTION_ROLE_PROMPT + "\n\n" + task,
                }
            ]
        })

        raw = result["messages"][-1].content.strip()

        print("\nRAW RESPONSE:")
        print(raw)

        parsed = extract_json(raw)

        print(f"========== END STAGE: {stage_name} | status=success ==========")
        return parsed

    except Exception as e:
        print(f"========== END STAGE: {stage_name} | status=failed ==========")
        print(f"ERROR: {e}")
        raise


In [ ]:
from pathlib import Path
import json


def dd_basic_overview_node(state):
    dataset_path = state["dataset_path"]

    task = f"""
Этап: basic_overview

Путь к датасету:
{dataset_path}

Цель:
Сформировать надежный первичный обзор сырого датасета. Используй только значения, реально вычисленные из датасета.

Используй python_interpreter_tool, чтобы написать и выполнить короткий Python-код.

Твой код должен:
1. Прочитать CSV в локальную переменную df.
2. Вычислить напрямую через pandas:
   - n_rows: количество строк
   - n_columns: количество колонок
   - columns: список названий колонок в исходном порядке
   - dtypes: pandas dtype для каждой колонки в виде строки
   - missing_values_total: общее количество пропущенных значений во всем датасете
   - duplicate_rows: количество полностью дублирующихся строк
   - memory_usage_mb: общий объем памяти dataframe в мегабайтах, округленный до 4 знаков
3. Не выполнять предобработку данных.
4. Не изменять df.
5. Не выводить Python-код, markdown, комментарии, объяснения или отладочный текст.
6. Не возвращать полный dataframe.
7. Не хранить полный dataframe в памяти для следующих этапов.
8. Каждый следующий этап при необходимости заново прочитает датасет.

Правила проверки:
- Все числовые значения должны быть реально вычислены, а не оценены примерно.
- Длина columns должна быть равна n_columns.
- Ключи dtypes должны точно совпадать с columns.
- Верни только компактный валидный JSON.

Верни только JSON:
{{
  "stage": "basic_overview",
  "status": "success",
  "result": {{
    "n_rows": 0,
    "n_columns": 0,
    "columns": [],
    "dtypes": {{}},
    "missing_values_total": 0,
    "duplicate_rows": 0,
    "memory_usage_mb": 0.0
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("basic_overview", task)
        basic_overview = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        basic_overview_path = artifacts_dir / "basic_overview.json"

        with open(basic_overview_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "data_description_agent",
            "stage": "basic_overview",
            "status": "success",
            "summary": "Basic dataset overview completed.",
            "artifacts": {
                "basic_overview_path": str(basic_overview_path),
            },
        }

        return {
            "basic_overview": basic_overview,
            "artifacts": {
                **state.get("artifacts", {}),
                "basic_overview_path": str(basic_overview_path),
            },
            "logs": [new_log],
            "status": "running",
            "error": None,
        }

    except Exception as e:
        new_log = {
            "agent": "data_description_agent",
            "stage": "basic_overview",
            "status": "failed",
            "summary": "Basic overview failed.",
            "reason": str(e),
        }

        return {
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }

In [ ]:
from pathlib import Path
import json


def dd_domain_understanding_node(state):
    dataset_path = state["dataset_path"]
    basic = state.get("basic_overview", {})

    task = f"""
Этап: domain_understanding

Путь к датасету:
{dataset_path}

Результат предыдущего этапа:
{json.dumps(basic, ensure_ascii=False, indent=2)}

Цель:
Определить предметную область датасета и смысл одной строки, используя только названия колонок и маленькие примеры значений.

Используй python_interpreter_tool, чтобы написать и выполнить короткий Python-код.

Твой код должен:
1. Прочитать CSV в локальную переменную df.
2. Посмотреть только компактные доказательства:
   - df.head(3)
   - названия колонок
   - dtypes
   - до 3 непустых примеров значений из важных object/string колонок
3. Не выводить Python-код, markdown, комментарии, объяснения или отладочный текст.
4. Не выводить полный dataframe.
5. Не хранить полный dataframe в памяти для следующих этапов.

Определи и верни:
- domain_summary: одно короткое предложение о вероятной предметной области
- row_meaning: одно короткое предложение о том, что, вероятно, означает одна строка
- column_groups_description: объект, где ключи — смысловые группы, а значения — списки колонок
- confidence: "low", "medium" или "high"
- notes: короткий список консервативных оговорок, если вывод неочевиден

Важные правила:
- Не выполнять предобработку данных.
- Не изменять df.
- Основывать вывод только на названиях колонок и компактных примерах значений.
- Если предметная область непонятна, укажи это в notes и поставь confidence="low".
- Верни только компактный валидный JSON.

Верни только JSON:
{{
  "stage": "domain_understanding",
  "status": "success",
  "result": {{
    "domain_summary": "",
    "row_meaning": "",
    "column_groups_description": {{}},
    "confidence": "low",
    "notes": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("domain_understanding", task)
        domain_understanding = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        domain_understanding_path = artifacts_dir / "domain_understanding.json"

        with open(domain_understanding_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "data_description_agent",
            "stage": "domain_understanding",
            "status": "success",
            "summary": "Domain understanding completed.",
            "artifacts": {
                "domain_understanding_path": str(domain_understanding_path),
            },
        }

        return {
            "domain_understanding": domain_understanding,
            "artifacts": {
                **state.get("artifacts", {}),
                "domain_understanding_path": str(domain_understanding_path),
            },
            "logs": [new_log],
            "status": "running",
            "error": None,
        }

    except Exception as e:
        new_log = {
            "agent": "data_description_agent",
            "stage": "domain_understanding",
            "status": "failed",
            "summary": "Domain understanding failed.",
            "reason": str(e),
        }

        return {
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }

In [ ]:
from pathlib import Path
import json


def dd_schema_detection_node(state):
    dataset_path = state["dataset_path"]
    target_column = state.get("target_column")
    basic = state.get("basic_overview", {})
    domain = state.get("domain_understanding", {})

    task = f"""
Этап: schema_detection

Путь к датасету:
{dataset_path}

Целевая колонка от пользователя:
{target_column}

Результаты предыдущих этапов:
basic_overview:
{json.dumps(basic, ensure_ascii=False, indent=2)}

domain_understanding:
{json.dumps(domain, ensure_ascii=False, indent=2)}

Цель:
Определить схему сырого датасета, целевую колонку и тип ML-задачи, если это возможно.

Используй python_interpreter_tool, чтобы написать и выполнить короткий Python-код.

Твой код должен:
1. Прочитать CSV в локальную переменную df.
2. Для каждой колонки вычислить:
   - pandas dtype
   - количество уникальных непустых значений
   - количество пропущенных значений
   - долю пропущенных значений
   - для object/string колонок: среднюю длину строки, максимальную длину строки и до 3 непустых примеров значений
3. Определить группы схемы:
   - numeric
   - categorical
   - text
   - datetime
   - boolean_like
   - id
   - target
4. Определить task_type:
   - "regression", если target числовой и непрерывный
   - "binary_classification", если target имеет ровно 2 различных непустых значения
   - "multiclass_classification", если target имеет больше 2 дискретных классов
   - "unknown", если target отсутствует или неясен

Правила классификации схемы:
- Колонка должна входить максимум в одну feature-группу, кроме отдельного отслеживания target.
- target должен содержать только выбранную целевую колонку, если она существует.
- Если target_column отсутствует в df, установи target_column в null и task_type в "unknown".
- Если target_column равен None, определяй target только если он очевиден по названиям колонок: target, label, y, price, outcome, class. Иначе используй null.
- ID-подобные колонки, например id, *_id, user_id, host_id, listing_id, должны быть id, а не numeric.
- Numeric — это int/float колонки, которые не являются id-like, boolean-like или target.
- Datetime применяется только к object/string/datetime колонкам. НЕ классифицируй numeric колонки как datetime.
- Boolean-like колонки имеют только boolean-подобные значения: true/false, yes/no, 0/1.
- Categorical — короткие string/object колонки с повторяющимися значениями и низкой или средней кардинальностью.
- Text — длинные свободные строки, описания, естественный язык, списки или высококардинальные строки.
- Колонки вроде name, description, overview, about, amenities, comments, review, summary считаются text, если содержат фразы или списки.

Важные правила:
- Не выполнять предобработку данных.
- Не изменять df.
- Не выводить Python-код, markdown, комментарии, объяснения или отладочный текст.
- Не возвращать полный dataframe.
- Не хранить полный dataframe в памяти для следующих этапов.
- Верни только компактный валидный JSON.

Верни только JSON:
{{
  "stage": "schema_detection",
  "status": "success",
  "result": {{
    "target_column": null,
    "task_type": "unknown",
    "schema": {{
      "numeric": [],
      "categorical": [],
      "text": [],
      "datetime": [],
      "boolean_like": [],
      "id": [],
      "target": []
    }},
    "column_profile": {{}},
    "schema_notes": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("schema_detection", task)
        schema_detection = parsed.get("result", {})

        detected_target_column = schema_detection.get("target_column")
        detected_task_type = schema_detection.get("task_type")
        detected_schema = schema_detection.get("schema", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        schema_detection_path = artifacts_dir / "schema_detection.json"

        with open(schema_detection_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "data_description_agent",
            "stage": "schema_detection",
            "status": "success",
            "summary": "Schema detection completed.",
            "artifacts": {
                "schema_detection_path": str(schema_detection_path),
            },
        }

        return {
            "schema_detection": schema_detection,

            # полезно сохранить наверх внутри DataDescriptionState
            "target_column": detected_target_column,
            "task_type": detected_task_type,
            "schema": detected_schema,

            "artifacts": {
                **state.get("artifacts", {}),
                "schema_detection_path": str(schema_detection_path),
            },
            "logs": [new_log],
            "status": "running",
            "error": None,
        }

    except Exception as e:
        new_log = {
            "agent": "data_description_agent",
            "stage": "schema_detection",
            "status": "failed",
            "summary": "Schema detection failed.",
            "reason": str(e),
        }

        return {
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }

In [ ]:
from pathlib import Path
import json


def dd_data_quality_node(state):
    dataset_path = state["dataset_path"]
    schema_detection = state.get("schema_detection", {})

    task = f"""
Этап: data_quality

Путь к датасету:
{dataset_path}

Результат schema_detection:
{json.dumps(schema_detection, ensure_ascii=False, indent=2)}

Цель:
Измерить проблемы качества данных в сыром датасете, используя только вычисленные значения.

Используй python_interpreter_tool, чтобы написать и выполнить короткий Python-код.

Твой код должен:
1. Прочитать CSV в локальную переменную df.
2. Использовать schema["numeric"] из переданного результата schema_detection.
3. Вычислить:
   - missing_values: количество пропусков по колонкам
   - missing_values_total: общее количество пропусков в датасете
   - missing_ratio_by_column: доля пропусков по колонкам, округленная до 6 знаков
   - duplicate_rows: количество полностью дублирующихся строк
   - constant_columns: колонки ровно с 1 уникальным непустым значением
   - high_cardinality_columns: колонки, где число уникальных непустых значений больше 90% от числа строк
   - high_missing_columns: колонки, где доля пропусков больше 0.5
   - numeric_outliers_iqr: количество выбросов для numeric-колонок из схемы по правилу IQR
4. Правило IQR для выбросов:
   - Q1 = 25-й процентиль
   - Q3 = 75-й процентиль
   - IQR = Q3 - Q1
   - lower = Q1 - 1.5 * IQR
   - upper = Q3 + 1.5 * IQR
   - выбросы — значения меньше lower или больше upper

Важные правила:
- Не выполнять предобработку данных.
- Не изменять df.
- Не выводить Python-код, markdown, комментарии, объяснения или отладочный текст.
- Не возвращать полный dataframe.
- Не хранить полный dataframe в памяти для следующих этапов.
- Если schema["numeric"] пустой, верни пустой словарь numeric_outliers_iqr.
- Значения в финальном JSON должны быть точно скопированы из переменных, вычисленных через python_interpreter_tool.
- Не оценивай приблизительно и не придумывай значения.
- Верни только компактный валидный JSON.

Верни только JSON:
{{
  "stage": "data_quality",
  "status": "success",
  "result": {{
    "missing_values": {{}},
    "missing_values_total": 0,
    "missing_ratio_by_column": {{}},
    "duplicate_rows": 0,
    "constant_columns": [],
    "high_cardinality_columns": [],
    "high_missing_columns": [],
    "numeric_outliers_iqr": {{}}
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("data_quality", task)
        data_quality = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        data_quality_path = artifacts_dir / "data_quality.json"

        with open(data_quality_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "data_description_agent",
            "stage": "data_quality",
            "status": "success",
            "summary": "Data quality analysis completed.",
            "artifacts": {
                "data_quality_path": str(data_quality_path),
            },
        }

        return {
            "data_quality": data_quality,
            "artifacts": {
                **state.get("artifacts", {}),
                "data_quality_path": str(data_quality_path),
            },
            "logs": [new_log],
            "status": "running",
            "error": None,
        }

    except Exception as e:
        new_log = {
            "agent": "data_description_agent",
            "stage": "data_quality",
            "status": "failed",
            "summary": "Data quality analysis failed.",
            "reason": str(e),
        }

        return {
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }

In [ ]:
from pathlib import Path
import json


def dd_statistics_node(state):
    dataset_path = state["dataset_path"]
    schema_detection = state.get("schema_detection", {})

    task = f"""
Этап: statistics

Путь к датасету:
{dataset_path}

Результат schema_detection:
{json.dumps(schema_detection, ensure_ascii=False, indent=2)}

Цель:
Посчитать компактные статистические метаданные для найденной схемы и целевой колонки.

Используй python_interpreter_tool, чтобы написать и выполнить короткий Python-код.

Твой код должен:
1. Прочитать CSV в локальную переменную df.
2. Использовать schema и target_column из переданного результата schema_detection.
3. Для числовых колонок вычислить компактные метаданные:
   - count, mean, std, min, median, max для каждой numeric-колонки
4. Для категориальных колонок вычислить компактные метаданные:
   - количество уникальных значений
   - самое частое значение
   - частота самого частого значения
5. Для текстовых колонок вычислить компактные метаданные:
   - количество непустых значений
   - средняя длина строки
   - максимальная длина строки
6. Для datetime-колонок вычислить компактные метаданные:
   - минимальная дата
   - максимальная дата
   - количество непустых значений
7. Для target_column, если она существует, вычислить:
   - target_type
   - количество пропусков
   - количество уникальных значений
   - распределение для classification target
   - summary statistics для regression target
8. Сохранить подробную статистику в JSON-артефакт этапа, вернув ее внутри result.

Важные правила:
- Не выполнять предобработку данных.
- Не изменять df.
- Не выводить Python-код, markdown, комментарии, объяснения или отладочный текст.
- Не возвращать полный dataframe.
- Не хранить полный dataframe в памяти для следующих этапов.
- Округлять float до 6 знаков.
- Использовать null для значений, которые невозможно вычислить.
- Верни только компактный валидный JSON.

Верни только JSON:
{{
  "stage": "statistics",
  "status": "success",
  "result": {{
    "numeric_columns_count": 0,
    "categorical_columns_count": 0,
    "text_columns_count": 0,
    "datetime_columns_count": 0,
    "target_column": null,
    "numeric_summary": {{}},
    "categorical_summary": {{}},
    "text_summary": {{}},
    "datetime_summary": {{}},
    "target_summary": {{}},
    "statistics_note": "Подробная статистика была рассчитана по сырому датасету."
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("statistics", task)
        statistics = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        statistics_path = artifacts_dir / "statistics.json"

        with open(statistics_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        statistics = {
            **statistics,
            "statistics_path": str(statistics_path),
        }

        new_log = {
            "agent": "data_description_agent",
            "stage": "statistics",
            "status": "success",
            "summary": "Statistics stage completed.",
            "artifacts": {
                "statistics_path": str(statistics_path),
            },
        }

        return {
            "statistics": statistics,
            "artifacts": {
                **state.get("artifacts", {}),
                "statistics_path": str(statistics_path),
            },
            "logs": [new_log],
            "status": "running",
            "error": None,
        }

    except Exception as e:
        new_log = {
            "agent": "data_description_agent",
            "stage": "statistics",
            "status": "failed",
            "summary": "Statistics calculation failed.",
            "reason": str(e),
        }

        return {
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }

In [ ]:
from pathlib import Path
import json


def dd_build_final_result_node(state):
    if state.get("error"):
        return {
            "logs": [{
                "agent": "data_description_agent",
                "stage": "build_final_result",
                "status": "skipped",
                "summary": "Final result build was skipped because previous stage failed.",
                "reason": state.get("error"),
            }],
        }

    basic = state.get("basic_overview", {})
    domain = state.get("domain_understanding", {})
    schema_detection = state.get("schema_detection", {})
    data_quality = state.get("data_quality", {})
    statistics = state.get("statistics", {})
    previous_artifacts = state.get("artifacts", {})

    required_basic_keys = ["n_rows", "n_columns", "columns", "dtypes"]
    missing_basic_keys = [key for key in required_basic_keys if key not in basic]

    if missing_basic_keys:
        error = f"Cannot build final result. Missing basic_overview keys: {missing_basic_keys}"
        return {
            "error": error,
            "logs": [{
                "agent": "data_description_agent",
                "stage": "build_final_result",
                "status": "failed",
                "summary": "Final result build failed.",
                "reason": error,
            }],
        }

    schema = schema_detection.get("schema", {})

    for key in ["numeric", "categorical", "text", "datetime"]:
        schema.setdefault(key, [])

    for key in ["boolean_like", "id", "target"]:
        schema.setdefault(key, [])

    target_column = schema_detection.get("target_column")
    task_type = schema_detection.get("task_type", "unknown")

    if target_column is None:
        task_type = "unknown"

    artifacts_dir = Path("artifacts/data_description")
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    summary_path = artifacts_dir / "data_description_summary.json"
    eda_artifacts_path = artifacts_dir / "eda_artifacts.json"
    report_path = artifacts_dir / "data_description_report.md"

    statistics_path = statistics.get(
        "statistics_path",
        previous_artifacts.get("statistics_path"),
    )

    artifacts = {
        **previous_artifacts,
        "data_description_summary_path": str(summary_path),
        "eda_artifacts_path": str(eda_artifacts_path),
        "data_description_report_path": str(report_path),
        "statistics_path": statistics_path,
        "eda_stages_dir": "artifacts/data_description/stages",
    }

    data_summary = {
        "n_rows": basic.get("n_rows"),
        "n_columns": basic.get("n_columns"),
        "domain_summary": domain.get("domain_summary"),
        "row_meaning": domain.get("row_meaning"),
        "missing_values_total": data_quality.get("missing_values_total"),
        "duplicate_rows": data_quality.get("duplicate_rows"),
        "statistics_path": statistics_path,
        "eda_artifacts_path": str(eda_artifacts_path),
    }

    final_result = {
        "agent": "data_description_agent",
        "status": "success",
        "skipped": False,
        "summary": (
            f"Dataset contains {basic.get('n_rows')} rows and "
            f"{basic.get('n_columns')} columns. Task type: {task_type}."
        ),
        "decisions": {
            "domain_summary": domain.get("domain_summary"),
            "row_meaning": domain.get("row_meaning"),
            "target_column": target_column,
            "task_type": task_type,
            "schema": schema,
            "data_quality_summary": {
                "missing_values_total": data_quality.get("missing_values_total"),
                "duplicate_rows": data_quality.get("duplicate_rows"),
                "constant_columns_count": len(data_quality.get("constant_columns", [])),
                "high_cardinality_columns_count": len(data_quality.get("high_cardinality_columns", [])),
            },
        },
        "artifacts": artifacts,
        "next_input": {
            "target_column": target_column,
            "task_type": task_type,
            "schema": schema,
            "data_summary": data_summary,
        },
        "reason": None,
    }

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(final_result, f, ensure_ascii=False, indent=2)

    eda_artifacts = {
        "basic_overview": basic,
        "domain_understanding": domain,
        "schema_detection": schema_detection,
        "data_quality": data_quality,
        "statistics": statistics,
        "stage_artifact_paths": {
            key: value
            for key, value in artifacts.items()
            if key.endswith("_path") or key.endswith("_dir")
        },
    }

    with open(eda_artifacts_path, "w", encoding="utf-8") as f:
        json.dump(eda_artifacts, f, ensure_ascii=False, indent=2)

    report_md = (
        "# Data Description Report\n\n"
        "## Summary\n"
        f"{final_result.get('summary')}\n\n"
        "## Domain\n"
        f"{domain.get('domain_summary')}\n\n"
        "## Row meaning\n"
        f"{domain.get('row_meaning')}\n\n"
        "## Target\n"
        f"- Target column: {target_column}\n"
        f"- Task type: {task_type}\n\n"
        "## Dataset shape\n"
        f"- Rows: {basic.get('n_rows')}\n"
        f"- Columns: {basic.get('n_columns')}\n\n"
        "## Data quality summary\n"
        f"- Total missing values: {data_quality.get('missing_values_total')}\n"
        f"- Duplicate rows: {data_quality.get('duplicate_rows')}\n"
        f"- Constant columns count: {len(data_quality.get('constant_columns', []))}\n"
        f"- High-cardinality columns count: {len(data_quality.get('high_cardinality_columns', []))}\n\n"
        "## Schema\n"
        "```json\n"
        f"{json.dumps(schema, ensure_ascii=False, indent=2)}\n"
        "```\n\n"
        "## Artifacts\n"
        f"- Summary: {summary_path}\n"
        f"- EDA artifacts: {eda_artifacts_path}\n"
        f"- Statistics: {statistics_path}\n"
        f"- Stage artifacts dir: {artifacts.get('eda_stages_dir')}\n"
    )

    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_md)

    print("\nRAW RESPONSE: build_final_result")
    print(json.dumps(final_result, ensure_ascii=False, indent=2))
    print("END RAW RESPONSE\n")

    return {
        "final_result": final_result,
        "artifacts": artifacts,
        "logs": [{
            "agent": "data_description_agent",
            "stage": "build_final_result",
            "status": "success",
            "summary": "Final compact data description result was built and saved.",
            "artifacts": {
                "data_description_summary_path": str(summary_path),
                "eda_artifacts_path": str(eda_artifacts_path),
                "data_description_report_path": str(report_path),
            },
        }],
    }

In [ ]:
def route_after_dd_stage(state):
    if state.get("error"):
        return "stop"
    return "continue"

In [ ]:
from langgraph.graph import StateGraph, START, END


def build_data_description_graph():
    workflow = StateGraph(DataDescriptionState)

    workflow.add_node("basic_overview", dd_basic_overview_node)
    workflow.add_node("domain_understanding", dd_domain_understanding_node)
    workflow.add_node("schema_detection", dd_schema_detection_node)
    workflow.add_node("data_quality", dd_data_quality_node)
    workflow.add_node("statistics", dd_statistics_node)
    workflow.add_node("build_final_result", dd_build_final_result_node)

    workflow.add_edge(START, "basic_overview")

    workflow.add_conditional_edges(
        "basic_overview",
        route_after_dd_stage,
        {"continue": "domain_understanding", "stop": END},
    )

    workflow.add_conditional_edges(
        "domain_understanding",
        route_after_dd_stage,
        {"continue": "schema_detection", "stop": END},
    )

    workflow.add_conditional_edges(
        "schema_detection",
        route_after_dd_stage,
        {"continue": "data_quality", "stop": END},
    )

    workflow.add_conditional_edges(
        "data_quality",
        route_after_dd_stage,
        {"continue": "statistics", "stop": END},
    )

    workflow.add_conditional_edges(
        "statistics",
        route_after_dd_stage,
        {"continue": "build_final_result", "stop": END},
    )

    workflow.add_conditional_edges(
        "build_final_result",
        route_after_dd_stage,
        {"continue": END, "stop": END},
    )

    return workflow.compile()

In [ ]:
def data_description_agent_node(state):
    dataset_path = state.get("dataset_path")
    target_column = state.get("target_column")

    if not dataset_path:
        log_record = {
            "agent": "data_description_agent",
            "status": "failed",
            "skipped": False,
            "summary": "Dataset path is missing.",
            "decisions": {},
            "artifacts": {},
            "next_input": {},
            "reason": "dataset_path is None",
        }

        return {
            "current_step": "data_description_agent",
            "error": "dataset_path is missing",
            "status": "failed",
            "logs": [log_record],
        }

    try:
        data_description_app = build_data_description_graph()

        dd_state = data_description_app.invoke(
            {
                "dataset_path": dataset_path,
                "target_column": target_column,
                "artifacts_dir": "artifacts/data_description",
                "logs": [],
                "error": None,
                "artifacts": {},
                "status": "running",
                "next_input": {},
            },
            {"recursion_limit": 20},
        )

        final_result = dd_state.get("final_result", {})

        if dd_state.get("error") or not final_result:
            error = dd_state.get("error") or "Data Description subgraph did not produce final_result."

            log_record = {
                "agent": "data_description_agent",
                "status": "failed",
                "skipped": False,
                "summary": "Data Description subgraph failed.",
                "decisions": {},
                "artifacts": dd_state.get("artifacts", {}),
                "next_input": {},
                "reason": error,
                "subgraph_logs": dd_state.get("logs", []),
            }

            return {
                "current_step": "data_description_agent",
                "error": error,
                "status": "failed",
                "artifacts": {
                    **state.get("artifacts", {}),
                    **dd_state.get("artifacts", {}),
                },
                "logs": [log_record],
            }

        next_input = final_result.get("next_input", {})
        decisions = final_result.get("decisions", {})
        artifacts = final_result.get("artifacts", {})

        schema = next_input.get("schema") or decisions.get("schema", {})
        data_summary = next_input.get("data_summary", {})

        detected_target_column = next_input.get("target_column", target_column)
        detected_task_type = next_input.get("task_type", state.get("task_type"))

        log_record = {
            "agent": "data_description_agent",
            "status": final_result.get("status", "success"),
            "skipped": final_result.get("skipped", False),
            "summary": final_result.get("summary", "Data description completed."),
            "decisions": decisions,
            "artifacts": artifacts,
            "next_input": next_input,
            "reason": final_result.get("reason"),
            "subgraph_logs": dd_state.get("logs", []),
        }

        return {
            "current_step": "data_description_agent",

            # Main detected task info
            "target_column": detected_target_column,
            "task_type": detected_task_type,
            "schema": schema,
            "data_summary": data_summary,

            # Important: this is the input for Data Preparation Agent
            "next_input": next_input,

            # Artifacts
            "artifacts": {
                **state.get("artifacts", {}),
                **artifacts,
            },

            # Logs are accumulated by operator.add in AgentState
            "logs": [log_record],

            # Clear previous orchestrator decision after executing the selected agent
            "next_action": None,
            "orchestration_decision": {},

            "status": "running",
            "error": None,
        }

    except Exception as e:
        log_record = {
            "agent": "data_description_agent",
            "status": "failed",
            "skipped": False,
            "summary": "Data Description subgraph failed.",
            "decisions": {},
            "artifacts": {},
            "next_input": {},
            "reason": str(e),
        }

        return {
            "current_step": "data_description_agent",
            "error": str(e),
            "status": "failed",
            "logs": [log_record],
        }